# Simple Dataset Generation with Exposure

This notebook demonstrates the simplest workflow to generate a synthetic insurance dataset with exposure periods and claims.

## Step 1: Import Libraries

Import the necessary modules from the claims simulator package.

In [ ]:
import numpy as np
import pandas as pd
import claimsimulator as csim

## Step 2: Define Features

Specify features to generate as a list of `Feature` dataclasses.  
Each feature needs a name and a distribution.

In [ ]:
# Define feature specifications
feature_spec = [
    csim.Feature('age', csim.Normal(loc=45, scale=15)),
    csim.Feature('annual_income', csim.Normal(loc=60000, scale=20000)),
    csim.Feature('years_licensed', csim.Normal(loc=20, scale=10)),
    csim.Feature('prior_claims', csim.Poisson(lam=0.5)),
]

# Create feature generator
feature_gen = csim.FeatureDefinition(feature_spec)

## Step 3: Generate Features

Create a dataset with the specified features for 1,000 policyholders.

In [11]:
# Generate 1,000 policyholders
features_df = feature_gen.generate(n_samples=1000, random_seed=42)

# Display the first few rows
print(f"Generated {len(features_df)} policyholders")
print(f"\nColumns: {list(features_df.columns)}")
print(f"\nFirst 5 rows:")
features_df.head()

Generated 1000 policyholders

Columns: ['age', 'annual_income', 'years_licensed', 'prior_claims']

First 5 rows:


,age,annual_income,years_licensed,prior_claims
0,52.450712,87987.108732,13.248217,2
1,42.926035,78492.673658,18.554813,1
2,54.715328,61192.607398,12.075801,0
3,67.845448,47061.264446,16.920385,0
4,41.487699,73964.466272,1.063853,0


## Step 4: Build Latent Risk Variable

Create a composite risk score by combining features with specific weights using a `FormulaFeature`.

In [ ]:
# Extend the feature spec with a formula-based risk score
feature_spec_with_risk = feature_spec + [
    csim.FormulaFeature(
        name='risk_score',
        formula='{beta_age} * age + {beta_income} * annual_income '
                '+ {beta_exp} * years_licensed + {beta_claims} * prior_claims',
        parameters={
            'beta_age': 0.02,              # Older drivers have higher risk
            'beta_income': -0.00001,       # Higher income = lower risk
            'beta_exp': -0.01,             # More experience = lower risk
            'beta_claims': 0.3,            # Past claims = higher risk
        },
    ),
]

# Generate features + risk in one call
features_with_risk = csim.FeatureDefinition(feature_spec_with_risk).generate(
    n_samples=1000, random_seed=42
)

print(f"Added latent risk variable: risk_score")
print(f"\nRisk score summary:")
features_with_risk['risk_score'].describe()

Added latent risk variable: risk_score

Risk score summary:


count    1000.000000
mean        0.235349
std         0.423030
min        -1.003009
25%        -0.051213
50%         0.217653
75%         0.513911
max         1.689724
Name: risk_score, dtype: float64

## Step 5: Transform Risk Score to Claim Rate

Transform the latent risk score into a positive claim rate using an exponential function.

In [13]:
# Convert risk score to claim rate using exponential transformation
# This ensures positive rates: rate = exp(risk_score)
risk_df = features_with_risk.copy()
risk_df['rate'] = np.exp(risk_df['risk_score'])

# Add contract duration (1 year for all policies)
risk_df['contract_duration'] = 1.0

print(f"Calculated claim rate from risk score")
print(f"\nClaim rate summary:")
risk_df['rate'].describe()

Calculated claim rate from risk score

Claim rate summary:


count    1000.000000
mean        1.386841
std         0.640703
min         0.366774
25%         0.950076
50%         1.243157
75%         1.671818
max         5.417985
Name: rate, dtype: float64

## Step 6: Simulate Claims with Exposure

Generate the final dataset with exposure periods and claim indicators.

In [ ]:
# Initialize claims simulator
simulator = csim.ClaimsSimulator(
    generator='Poisson',
    param_columns={'rate': 'rate'},
    time_to_simulate='contract_duration',
    max_exposure=1.0,
    random_seed=42
)

# Simulate claims
claims_df = simulator.simulate(risk_df)

print(f"Generated {len(claims_df)} rows from {len(risk_df)} contracts")
print(f"\nColumns in final dataset: {list(claims_df.columns)}")
print(f"\nTotal claims: {claims_df['claim'].sum()}")
print(f"Total exposure: {claims_df['exposure'].sum():.2f} years")

Generated 2395 rows from 1000 contracts

Columns in final dataset: ['age', 'annual_income', 'years_licensed', 'prior_claims', 'risk_score', 'rate', 'contract_duration', 'exposure', 'claim', 'start_time', 'end_time', 'contract_id']

Total claims: 1395
Total exposure: 1000.00 years


## Step 7: View the Final Dataset

Display the structure of the final dataset with exposure and claims.

In [15]:
# Show first 10 rows
print("First 10 rows of the dataset:")
claims_df.head(10)

First 10 rows of the dataset:


,age,annual_income,years_licensed,prior_claims,risk_score,rate,contract_duration,exposure,claim,start_time,end_time,contract_id
0,52.450712,87987.108732,13.248217,2.0,0.636661,1.890159,1.0,0.248269,1,0.000000,0.248269,0
1,52.450712,87987.108732,13.248217,2.0,0.636661,1.890159,1.0,0.751731,0,0.248269,1.000000,0
2,42.926035,78492.673658,18.554813,1.0,0.188046,1.206889,1.0,1.000000,0,0.000000,1.000000,1
3,54.715328,61192.607398,12.075801,0.0,0.361622,1.435657,1.0,0.635906,1,0.000000,0.635906,2
4,54.715328,61192.607398,12.075801,0.0,0.361622,1.435657,1.0,0.118151,1,0.635906,0.754057,2
5,54.715328,61192.607398,12.075801,0.0,0.361622,1.435657,1.0,0.118131,1,0.754057,0.872189,2
6,54.715328,61192.607398,12.075801,0.0,0.361622,1.435657,1.0,0.041680,1,0.872189,0.913869,2
7,54.715328,61192.607398,12.075801,0.0,0.361622,1.435657,1.0,0.086131,0,0.913869,1.000000,2
8,67.845448,47061.264446,16.920385,0.0,0.717092,2.048469,1.0,0.448668,1,0.000000,0.448668,3
9,67.845448,47061.264446,16.920385,0.0,0.717092,2.048469,1.0,0.551332,0,0.448668,1.000000,3


## Step 8: Inspect Sample Contracts

Look at a few individual contracts to understand the exposure structure.

In [16]:
# Show first 3 contracts
print("First 3 complete contracts:")
claims_df[claims_df['contract_id'].isin([0, 1, 2])]

First 3 complete contracts:


,age,annual_income,years_licensed,prior_claims,risk_score,rate,contract_duration,exposure,claim,start_time,end_time,contract_id
0,52.450712,87987.108732,13.248217,2.0,0.636661,1.890159,1.0,0.248269,1,0.000000,0.248269,0
1,52.450712,87987.108732,13.248217,2.0,0.636661,1.890159,1.0,0.751731,0,0.248269,1.000000,0
2,42.926035,78492.673658,18.554813,1.0,0.188046,1.206889,1.0,1.000000,0,0.000000,1.000000,1
3,54.715328,61192.607398,12.075801,0.0,0.361622,1.435657,1.0,0.635906,1,0.000000,0.635906,2
4,54.715328,61192.607398,12.075801,0.0,0.361622,1.435657,1.0,0.118151,1,0.635906,0.754057,2
5,54.715328,61192.607398,12.075801,0.0,0.361622,1.435657,1.0,0.118131,1,0.754057,0.872189,2
6,54.715328,61192.607398,12.075801,0.0,0.361622,1.435657,1.0,0.041680,1,0.872189,0.913869,2
7,54.715328,61192.607398,12.075801,0.0,0.361622,1.435657,1.0,0.086131,0,0.913869,1.000000,2


## Step 9: Save the Dataset

Save the final dataset to disk preserving the schema and data types.

In [ ]:
import os

# Create data directory if it doesn't exist
os.makedirs('../data', exist_ok=True)

# Option 1: Pickle format (preserves exact dtypes, Python-only)
# Best for: Python-to-Python workflows, preserves all metadata
pickle_path = '../data/claims_dataset.pkl'
claims_df.to_pickle(pickle_path)
print(f"✓ Saved as pickle: {pickle_path}")
print(f"  Size: {os.path.getsize(pickle_path) / 1024:.1f} KB")

# Option 2: CSV format (human-readable, cross-platform)
# Best for: Sharing with non-Python tools, inspection
csv_path = '../data/claims_dataset.csv'
claims_df.to_csv(csv_path, index=False)
print(f"\n✓ Saved as CSV: {csv_path}")
print(f"  Size: {os.path.getsize(csv_path) / 1024:.1f} KB")

# Load back pickle to verify
claims_loaded = pd.read_pickle(pickle_path)
print(f"\n✓ Verified: Loaded {len(claims_loaded)} rows")
print(f"  Schema preserved: {claims_loaded.dtypes.equals(claims_df.dtypes)}")
print(f"\nDataframe info:")
claims_loaded.info()

## Summary

This notebook demonstrated the complete workflow:

1. **Define features** - Specify distributions for policyholder characteristics
2. **Generate features** - Create synthetic policyholder data
3. **Build latent risk** - Combine features into a composite risk score
4. **Transform to claim rate** - Apply exponential transformation: rate = exp(risk_score)
5. **Simulate claims** - Generate exposure periods and claim events
6. **Save dataset** - Persist to disk in pickle and CSV formats

The final dataset includes:
- **exposure**: Time period each row represents (in years)
- **claim**: Binary indicator (1 = claim occurred, 0 = no claim)
- **contract_id**: Identifies unique policies
- **rate**: Expected claims per year for each policy
- All original features and risk scores

**Saved formats:**
- **Pickle (`.pkl`)**: Preserves exact dtypes and schema, ideal for Python workflows
- **CSV (`.csv`)**: Human-readable format for sharing and inspection

This format is ready for modeling claim frequency using GLM or survival models.